In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1GsCs8qFwN-EJOe8y3Mk1_YAuVtWWhfvv", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_01_intro.mp3"))

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Notebook 2: Structured Error Responses for Agent Tools

**Claude Certified Architect Prep** | Pod 2: Tool Design & MCP Integration

---

In Notebook 1, we learned how to write tool descriptions that help agents pick the right tool. But what happens when the right tool is selected... and it **fails**?

An agent with no structured error information is like a customer support rep who only hears "something went wrong" on the phone -- they cannot help, cannot retry, and cannot explain.

In this notebook, you will build a **complete error taxonomy** for MCP tool responses. You will implement the `isError` flag, classify errors into four categories, decide which are retryable, and build multi-agent systems that know when to retry locally versus escalate.

**What you will build:**
- A structured `ToolErrorResponse` with categories, retry logic, and customer-facing messages
- A flowchart visualization of agent error-handling strategy
- A multi-agent system with local recovery and propagation patterns

**Prerequisites:** Notebook 1 (Crafting Effective Tool Descriptions)

**Estimated time:** 60 minutes

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/tool-design-mcp/practice/2/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
# Setup - install dependencies
# All code in this notebook is self-contained with mock tool calls (no API keys needed)

!pip install -q matplotlib numpy

import json
import time
import random
from enum import Enum
from dataclasses import dataclass, field, asdict
from typing import Optional, Any, Dict, List, Callable
from datetime import datetime

print("Setup complete! All imports ready.")
print("No API keys required -- this notebook uses mock tool calls throughout.")

In [ ]:
#@title 🎧 Listen: Warmup Problem
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_warmup_problem.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---
## Section 1: Warm-Up -- The "Operation Failed" Problem

Let's start with a scenario every architect encounters. You have built a Customer Support Agent that uses a `process_refund` tool. A customer asks for a refund, the agent calls the tool, and gets back...

In [ ]:
# === The Problem: Unstructured Error Responses ===

def process_refund_v1(order_id: str, amount: float) -> dict:
    """A poorly designed tool that returns vague errors."""
    # Simulate various failure modes -- all equally unhelpful
    failures = [
        "Operation failed",
        "Error processing request",
        "Something went wrong",
        "Internal server error",
    ]
    return {"error": random.choice(failures)}


# Simulate what the agent sees
print("=" * 65)
print("SCENARIO: Customer asks for a $49.99 refund on order #12345")
print("=" * 65)

print("\n--- Agent calls process_refund tool ---")
result = process_refund_v1("ORD-12345", 49.99)
print(f"Tool response: {json.dumps(result, indent=2)}")

print("\n--- Agent's internal reasoning ---")
print("""
The tool returned an error, but I don't know:
  1. WHAT went wrong      -- Is the order invalid? Is the payment system down?
  2. WHETHER to retry      -- Will trying again help, or just waste time?
  3. WHAT to tell customer -- "Something went wrong" is not acceptable
  4. WHETHER to escalate   -- Should I try a different approach or give up?

I'm stuck. I'll tell the customer: "I'm sorry, I encountered an error
processing your refund. Please try again later."
""")

print("The customer is frustrated. The agent looks incompetent.")
print("And the real cause? The payment gateway had a 2-second timeout.")
print("A simple retry would have worked.")

In [ ]:
#@title 🎧 Listen: Structured Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_structured_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# === The Solution: Structured Error Responses ===

def process_refund_v2(order_id: str, amount: float) -> dict:
    """A well-designed tool with structured error information."""
    # Simulate the same timeout failure, but with structure
    return {
        "isError": True,
        "errorCategory": "transient",
        "errorCode": "PAYMENT_GATEWAY_TIMEOUT",
        "message": "Payment gateway did not respond within 5000ms",
        "isRetryable": True,
        "retryAfterMs": 2000,
        "customerMessage": "Your refund is taking a moment to process. Trying again now.",
        "context": {
            "gateway": "stripe",
            "timeout_ms": 5000,
            "order_id": order_id,
            "attempt": 1,
        },
    }


print("=" * 65)
print("SAME SCENARIO with structured error responses")
print("=" * 65)

print("\n--- Agent calls process_refund tool (v2) ---")
result = process_refund_v2("ORD-12345", 49.99)
print(f"Tool response: {json.dumps(result, indent=2)}")

print("\n--- Agent's internal reasoning (with structured errors) ---")
print("""
The tool returned isError=True with structured metadata. I know:
  1. CATEGORY: transient    -- This is a temporary infrastructure issue
  2. RETRYABLE: yes         -- I should try again
  3. RETRY AFTER: 2000ms    -- Wait 2 seconds before retrying
  4. CUSTOMER MSG: provided  -- I have a friendly message ready
  5. CONTEXT: Stripe timeout -- If retry fails, I can escalate with details

Action: Wait 2 seconds, retry, and tell the customer their refund
is being processed.
""")

print("The agent retries, succeeds, and the customer is happy.")
print("Same failure. Completely different outcome.")
print("\nKey insight: The difference between a stuck agent and a resilient")
print("agent is not better prompting -- it is STRUCTURED ERROR METADATA.")

In [ ]:
#@title 🎧 Listen: Error Categories
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_error_categories.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---
## Section 2: The isError Flag and Error Categories

The MCP (Model Context Protocol) specification includes an `isError` field on tool results. When `isError` is `true`, the model treats the content as an error rather than a successful result. But the spec leaves it to *you* to decide what structure that error content takes.

Here is the taxonomy we will use throughout this notebook. These four categories cover the vast majority of real-world tool failures:

| Category | Meaning | Agent Action | isRetryable | Example |
|---|---|---|---|---|
| **transient** | Temporary infrastructure failure | Retry with backoff | `True` | Timeout, rate limit, 503 |
| **validation** | Bad input from the agent | Fix input and retry | `True` (after fix) | Missing field, wrong format |
| **business** | Valid request, but business rules block it | Explain to user | `False` | Refund > order amount, account frozen |
| **permission** | Caller lacks authorization | Escalate | `False` | Insufficient role, expired token |

The key classification question is: **"If I send the exact same request in 5 minutes, will it succeed?"**
- **Yes** -> transient
- **No, but with different input** -> validation
- **No, never (policy)** -> business
- **No, never (authorization)** -> permission

In [ ]:
#@title 🎧 Listen: Factory Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_factory_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# === Core Data Structures: Error Category Taxonomy ===

class ErrorCategory(Enum):
    """The four canonical error categories for MCP tool responses."""
    TRANSIENT = "transient"       # Infrastructure flake -- retry will likely succeed
    VALIDATION = "validation"     # Bad input -- fix the request and retry
    BUSINESS = "business"         # Business rule violation -- explain to user
    PERMISSION = "permission"     # Authorization failure -- escalate


@dataclass
class ToolErrorResponse:
    """
    Structured error response for MCP tool calls.

    This wraps the MCP isError flag with a rich taxonomy that gives
    agents enough information to recover automatically.
    """
    isError: bool = True
    errorCategory: ErrorCategory = ErrorCategory.TRANSIENT
    errorCode: str = ""                     # Machine-readable code (e.g., "RATE_LIMITED")
    message: str = ""                       # Developer-facing detail
    isRetryable: bool = False               # Should the agent retry this exact call?
    retryAfterMs: Optional[int] = None      # How long to wait before retry (ms)
    maxRetries: int = 0                     # Max retry attempts suggested
    customerMessage: Optional[str] = None   # Safe message for end users
    context: Dict[str, Any] = field(default_factory=dict)  # Extra debugging info

    def to_mcp_response(self) -> dict:
        """Convert to MCP-compatible tool result format."""
        return {
            "isError": self.isError,
            "content": [
                {
                    "type": "text",
                    "text": json.dumps({
                        "errorCategory": self.errorCategory.value,
                        "errorCode": self.errorCode,
                        "message": self.message,
                        "isRetryable": self.isRetryable,
                        "retryAfterMs": self.retryAfterMs,
                        "maxRetries": self.maxRetries,
                        "customerMessage": self.customerMessage,
                        "context": self.context,
                    }, indent=2)
                }
            ]
        }


# === Factory functions for each category ===

def transient_error(code: str, message: str, retry_after_ms: int = 1000,
                    max_retries: int = 3, context: dict = None) -> ToolErrorResponse:
    """Create a transient (retryable infrastructure) error."""
    return ToolErrorResponse(
        errorCategory=ErrorCategory.TRANSIENT,
        errorCode=code,
        message=message,
        isRetryable=True,
        retryAfterMs=retry_after_ms,
        maxRetries=max_retries,
        customerMessage="This is taking a moment. Please hold on while I try again.",
        context=context or {},
    )


def validation_error(code: str, message: str, fix_hint: str = "",
                     context: dict = None) -> ToolErrorResponse:
    """Create a validation (bad input) error."""
    return ToolErrorResponse(
        errorCategory=ErrorCategory.VALIDATION,
        errorCode=code,
        message=message,
        isRetryable=True,  # Retryable after fixing input
        retryAfterMs=0,    # No wait needed -- fix input immediately
        maxRetries=2,
        customerMessage=None,  # Agent should fix silently, not tell user
        context={**(context or {}), "fix_hint": fix_hint},
    )


def business_error(code: str, message: str, customer_message: str,
                   context: dict = None) -> ToolErrorResponse:
    """Create a business rule violation error."""
    return ToolErrorResponse(
        errorCategory=ErrorCategory.BUSINESS,
        errorCode=code,
        message=message,
        isRetryable=False,      # Don't retry -- business rule won't change
        customerMessage=customer_message,
        context=context or {},
    )


def permission_error(code: str, message: str, required_role: str = "",
                     context: dict = None) -> ToolErrorResponse:
    """Create a permission/authorization error."""
    return ToolErrorResponse(
        errorCategory=ErrorCategory.PERMISSION,
        errorCode=code,
        message=message,
        isRetryable=False,  # Don't retry -- need different credentials
        customerMessage="I need to transfer you to someone with the right access for this.",
        context={**(context or {}), "required_role": required_role},
    )


# === Demo: Create one of each ===
print("=== Error Category Examples ===\n")

examples = [
    transient_error(
        "PAYMENT_GATEWAY_TIMEOUT",
        "Stripe API did not respond within 5000ms",
        retry_after_ms=2000,
        context={"gateway": "stripe"}
    ),
    validation_error(
        "INVALID_EMAIL_FORMAT",
        "Email 'john@' is not a valid email address",
        fix_hint="Ensure email has domain (e.g., john@example.com)",
    ),
    business_error(
        "REFUND_EXCEEDS_ORDER",
        "Refund amount $150.00 exceeds order total $99.99",
        "I can only process a refund up to the order total of $99.99. "
        "Would you like me to process that amount instead?",
        context={"order_total": 99.99, "requested_refund": 150.00},
    ),
    permission_error(
        "INSUFFICIENT_ROLE",
        "Agent role 'support_tier1' cannot process refunds over $500",
        required_role="support_tier2",
    ),
]

for err in examples:
    cat = err.errorCategory.value.upper()
    print(f"[{cat}] {err.errorCode}")
    print(f"  Message:     {err.message}")
    print(f"  Retryable:   {err.isRetryable}")
    if err.retryAfterMs:
        print(f"  RetryAfter:  {err.retryAfterMs}ms")
    else:
        print(f"  RetryAfter:  N/A")
    print(f"  Customer:    {err.customerMessage}")
    print()

print("=== MCP Wire Format (what the model actually sees) ===")
print(json.dumps(examples[0].to_mcp_response(), indent=2))

In [ ]:
#@title 🎧 Listen: Classify Exercise
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_classify_exercise.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

**Design notes on the four categories:**

- **Transient**: The only category where an *identical* retry makes sense. The infrastructure hiccuped -- same request, same params, just try again after a delay.
- **Validation**: Retryable, but the agent must *change the input*. An identical retry would fail again. The `fix_hint` field tells the agent what to correct.
- **Business**: Not retryable at all. The business rule is the final answer. The agent's job is to explain the constraint to the user and offer alternatives.
- **Permission**: Not retryable with the current credentials. The agent should escalate (transfer to a human, elevate permissions, or explain the limitation).

---
## Section 3: Guided Exercise -- Classify Errors

Below are 10 real-world error scenarios. For each one, classify it into one of the four categories and decide whether it is retryable.

**Instructions:** Read each scenario, fill in `category` and `is_retryable`, then run the solution cell to check your answers.

In [ ]:
# === Exercise: Classify These Errors ===
# For each scenario, set:
#   "category": one of "transient", "validation", "business", "permission"
#   "is_retryable": True or False

scenarios = [
    {
        "id": 1,
        "scenario": "A database query times out after 30 seconds due to high server load.",
        "error_message": "Query execution exceeded 30000ms timeout",
        "category": "???",       # <-- YOUR ANSWER
        "is_retryable": None,    # <-- YOUR ANSWER (True/False)
    },
    {
        "id": 2,
        "scenario": "Agent sends a date as '2024-13-45' which is not a valid calendar date.",
        "error_message": "Invalid date format: month 13 is out of range [1-12]",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 3,
        "scenario": "Customer tries to cancel an order that has already shipped and is in transit.",
        "error_message": "Order ORD-789 has status 'shipped' and cannot be cancelled",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 4,
        "scenario": "The agent's API bearer token expired 5 minutes ago.",
        "error_message": "Bearer token expired at 2024-03-15T10:00:00Z",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 5,
        "scenario": "A third-party shipping API returns HTTP 503 Service Unavailable.",
        "error_message": "Shipping provider returned 503: service temporarily unavailable",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 6,
        "scenario": "Agent tries to apply a 50% discount but the max allowed discount is 25%.",
        "error_message": "Discount 50% exceeds maximum allowed discount of 25% for this product tier",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 7,
        "scenario": "Agent sends order_id as integer 12345 but the API expects string format 'ORD-12345'.",
        "error_message": "Field 'order_id' expected string matching pattern 'ORD-\\d+', got integer 12345",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 8,
        "scenario": "Rate limiter kicks in -- 100 of 100 allowed requests per minute used.",
        "error_message": "Rate limit exceeded: 100/100 requests used. Retry after 45 seconds.",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 9,
        "scenario": "A Tier-1 support agent tries to view a customer's stored payment details.",
        "error_message": "Role 'support_tier1' does not have access to resource 'payment_details'",
        "category": "???",
        "is_retryable": None,
    },
    {
        "id": 10,
        "scenario": "Customer requests a refund but their account has an active fraud investigation hold.",
        "error_message": "Account ACC-456 has flag 'fraud_review'. All refund operations suspended.",
        "category": "???",
        "is_retryable": None,
    },
]

print("Classify each scenario: transient | validation | business | permission")
print("And decide: is_retryable = True or False\n")

for s in scenarios:
    print(f"  #{s['id']}: {s['scenario']}")
    print(f"      Error: \"{s['error_message']}\"")
    print(f"      Your answer: category={s['category']}, is_retryable={s['is_retryable']}")
    print()

print("Fill in your answers above, then run the next cell to check!")

In [ ]:
# === Solution: Error Classification Answers ===

solutions = [
    {"id": 1,  "category": "transient",   "is_retryable": True,
     "explanation": "Database timeout under high load is a temporary infrastructure issue. "
                    "The same query will likely succeed when load decreases."},

    {"id": 2,  "category": "validation",   "is_retryable": True,
     "explanation": "The date '2024-13-45' is malformed input from the agent. "
                    "Retryable after the agent fixes the date to a valid value."},

    {"id": 3,  "category": "business",     "is_retryable": False,
     "explanation": "The order has shipped -- this is a business state. No retry will change "
                    "the order's status. Agent should explain this and offer the return process."},

    {"id": 4,  "category": "permission",   "is_retryable": False,
     "explanation": "An expired token is an authorization failure. The agent cannot fix this "
                    "by retrying with the same token. Needs token refresh or escalation."},

    {"id": 5,  "category": "transient",    "is_retryable": True,
     "explanation": "HTTP 503 means the service is temporarily down. Classic transient "
                    "error -- retry with backoff."},

    {"id": 6,  "category": "business",     "is_retryable": False,
     "explanation": "The 25% discount cap is a business rule, not a bug. The agent should "
                    "offer the maximum 25% discount instead of retrying."},

    {"id": 7,  "category": "validation",   "is_retryable": True,
     "explanation": "Wrong data type/format for order_id. Agent should reformat as "
                    "'ORD-12345' and retry."},

    {"id": 8,  "category": "transient",    "is_retryable": True,
     "explanation": "Rate limiting is a transient infrastructure constraint. Wait 45 "
                    "seconds and retry. The error even tells you when."},

    {"id": 9,  "category": "permission",   "is_retryable": False,
     "explanation": "Tier-1 agents lack the role to access payment details. This "
                    "requires role elevation or transfer to Tier-2."},

    {"id": 10, "category": "business",     "is_retryable": False,
     "explanation": "A fraud hold is a business-level restriction on the ACCOUNT, not "
                    "a limitation of the agent's role. Even a Tier-2 agent with full "
                    "permissions would be blocked by this hold."},
]

print("=== SOLUTIONS ===\n")

for sol in solutions:
    print(f"  #{sol['id']}: {sol['category'].upper()} | Retryable: {sol['is_retryable']}")
    print(f"       {sol['explanation']}")
    print()

print("=" * 65)
print("COMMON MISTAKES:")
print("=" * 65)
print()
print("  #4 (expired token): Some classify as 'transient' because tokens can")
print("  be refreshed. But from the TOOL's perspective, the current request")
print("  cannot succeed with these credentials. The agent needs different")
print("  credentials, making it a permission issue.")
print()
print("  #10 (fraud hold): Some classify as 'permission'. But the fraud hold")
print("  is a business state on the ACCOUNT, not a limitation of the agent's")
print("  role. A Tier-2 agent with full permissions would also be blocked.")
print()
print("  #6 vs #9: Both say 'you cannot do this', but for different reasons.")
print("  #6 is a policy on the ACTION (discount cap). #9 is a restriction on")
print("  the CALLER (wrong role). That's the business vs permission distinction.")

In [ ]:
#@title 🎧 Listen: Flowchart
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_flowchart.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---
## Section 4: Error Response Flow Visualization

Now let's visualize how an agent should handle each error category. This flowchart shows the decision tree an agent follows when it receives an error from a tool.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis("off")
fig.patch.set_facecolor("white")

# --- Color palette ---
COLORS = {
    "start":      "#4A90D9",
    "decision":   "#F5A623",
    "transient":  "#7ED321",
    "validation": "#50C8E8",
    "business":   "#E85C8A",
    "permission": "#BD6BD9",
    "action":     "#F0F0F0",
    "success":    "#2ECC71",
}

def draw_box(ax, x, y, w, h, text, color, fontsize=9, fontweight="normal",
             text_color="white"):
    """Draw a rounded rectangle with centered text."""
    fancy = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.1",
        facecolor=color, edgecolor="gray", linewidth=1.2
    )
    ax.add_patch(fancy)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize,
            fontweight=fontweight, color=text_color, wrap=True,
            bbox=dict(boxstyle="round,pad=0.05", facecolor="none",
                      edgecolor="none"))

def draw_diamond(ax, x, y, w, h, text, color, fontsize=8):
    """Draw a diamond (decision node) with centered text."""
    diamond = plt.Polygon([
        [x, y + h/2], [x + w/2, y], [x, y - h/2], [x - w/2, y]
    ], facecolor=color, edgecolor="gray", linewidth=1.2)
    ax.add_patch(diamond)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize,
            fontweight="bold", color="white")

def draw_arrow(ax, x1, y1, x2, y2, label="", color="gray"):
    """Draw an arrow between two points with an optional label."""
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.5))
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx + 0.15, my + 0.1, label, fontsize=7, color=color,
                fontstyle="italic")

# === ROW 1: Tool Call ===
draw_box(ax, 7, 9.2, 3.5, 0.6, "Agent calls MCP Tool",
         COLORS["start"], fontsize=11, fontweight="bold")

# === ROW 2: isError check ===
draw_arrow(ax, 7, 8.9, 7, 8.3)
draw_diamond(ax, 7, 7.9, 2.5, 0.8, "isError = true?", COLORS["decision"])

# No branch (success)
draw_arrow(ax, 8.25, 7.9, 11.5, 7.9, label="No", color="#2ECC71")
draw_box(ax, 12.5, 7.9, 1.6, 0.55, "Success!\nUse result",
         COLORS["success"], fontsize=8, fontweight="bold")

# Yes branch (down)
draw_arrow(ax, 7, 7.5, 7, 6.9, label="Yes", color="#E74C3C")

# === ROW 3: Error category decision ===
draw_diamond(ax, 7, 6.5, 2.8, 0.8, "errorCategory?", COLORS["decision"])

# === Four branches ===

# -- TRANSIENT (leftmost) --
draw_arrow(ax, 5.6, 6.5, 2.5, 5.5, label="transient",
           color=COLORS["transient"])
draw_box(ax, 2.5, 5.1, 2.8, 0.7,
         "TRANSIENT\nWait retryAfterMs\nRetry (up to maxRetries)",
         COLORS["transient"], fontsize=8, fontweight="bold")
draw_arrow(ax, 2.5, 4.7, 2.5, 4.1)
draw_box(ax, 2.5, 3.75, 2.6, 0.55,
         "Tell customer:\n\"Working on it...\"",
         COLORS["action"], fontsize=7, text_color="black")
draw_arrow(ax, 2.5, 3.45, 2.5, 2.9)
draw_diamond(ax, 2.5, 2.55, 2.0, 0.6, "Retry\nsucceeded?",
             COLORS["decision"], fontsize=7)
draw_arrow(ax, 3.5, 2.55, 4.5, 2.55, label="Yes", color="#2ECC71")
draw_box(ax, 5.3, 2.55, 1.2, 0.45, "Done",
         COLORS["success"], fontsize=8, fontweight="bold")
draw_arrow(ax, 1.5, 2.55, 0.8, 2.55, label="No", color="#E74C3C")
draw_box(ax, 0.8, 2.1, 1.4, 0.45, "Escalate",
         COLORS["permission"], fontsize=8, fontweight="bold")

# -- VALIDATION (center-left) --
draw_arrow(ax, 6.3, 6.1, 5.2, 5.5, label="validation",
           color=COLORS["validation"])
draw_box(ax, 5.2, 5.1, 2.5, 0.7,
         "VALIDATION\nRead fix_hint\nCorrect input & retry",
         COLORS["validation"], fontsize=8, fontweight="bold")
draw_arrow(ax, 5.2, 4.7, 5.2, 4.1)
draw_box(ax, 5.2, 3.75, 2.4, 0.55,
         "Silent fix -- don't\nbother the customer",
         COLORS["action"], fontsize=7, text_color="black")

# -- BUSINESS (center-right) --
draw_arrow(ax, 7.7, 6.1, 8.8, 5.5, label="business",
           color=COLORS["business"])
draw_box(ax, 8.8, 5.1, 2.6, 0.7,
         "BUSINESS\nDon't retry\nExplain constraint",
         COLORS["business"], fontsize=8, fontweight="bold")
draw_arrow(ax, 8.8, 4.7, 8.8, 4.1)
draw_box(ax, 8.8, 3.75, 2.5, 0.55,
         "Tell customer:\ncustomerMessage\n+ offer alternatives",
         COLORS["action"], fontsize=7, text_color="black")

# -- PERMISSION (rightmost) --
draw_arrow(ax, 8.4, 6.5, 11.5, 5.5, label="permission",
           color=COLORS["permission"])
draw_box(ax, 11.5, 5.1, 2.4, 0.7,
         "PERMISSION\nDon't retry\nEscalate to human",
         COLORS["permission"], fontsize=8, fontweight="bold")
draw_arrow(ax, 11.5, 4.7, 11.5, 4.1)
draw_box(ax, 11.5, 3.75, 2.4, 0.55,
         "Transfer to agent\nwith required_role",
         COLORS["action"], fontsize=7, text_color="black")

# === Title ===
ax.text(7, 9.8, "Agent Error Handling Flowchart",
        ha="center", va="center", fontsize=14, fontweight="bold",
        color="#333333")

# === Legend ===
legend_y = 1.2
legend_items = [
    ("Transient: Retry with backoff", COLORS["transient"]),
    ("Validation: Fix input & retry", COLORS["validation"]),
    ("Business: Explain to user", COLORS["business"]),
    ("Permission: Escalate", COLORS["permission"]),
]
for i, (label, color) in enumerate(legend_items):
    x = 1.5 + i * 3.2
    ax.add_patch(mpatches.FancyBboxPatch(
        (x - 0.3, legend_y - 0.15), 0.4, 0.3,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="none"))
    ax.text(x + 0.4, legend_y, label, fontsize=8, va="center", color="#333")

plt.tight_layout()
plt.savefig("error_flow.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("\nFlowchart saved as 'error_flow.png'")

In [ ]:
#@title 🎧 Listen: Handler Exercise
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_handler_exercise.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

**Reading the flowchart:**

1. Every tool response is first checked for `isError`. If `false`, the result is used normally.
2. If `isError` is `true`, the agent reads `errorCategory` to decide its strategy:
   - **Transient** -- wait, retry, tell the customer to hang on
   - **Validation** -- silently fix the input and retry (the customer does not need to know the agent made a typo)
   - **Business** -- explain the constraint using `customerMessage` and offer alternatives
   - **Permission** -- transfer to a human or agent with the right role

Notice that **transient** and **validation** both involve retrying, but in fundamentally different ways:
- Transient = same request, just try again after a delay
- Validation = *modified* request with corrected input

---
## Section 5: TODO Exercise -- Build an MCP Error Handler

Now it is your turn. Implement the `MCPToolHandler` class that wraps any tool function with structured error handling. The handler should:

1. Call the tool function
2. Catch exceptions and classify them into error categories
3. Return a properly structured `ToolErrorResponse`
4. Support automatic retries for transient errors

The class skeleton is provided. Fill in the `TODO` sections.

In [ ]:
# === Custom Exception Types (simulate what real tool implementations raise) ===

class TransientError(Exception):
    """Raised when a temporary infrastructure issue occurs."""
    pass

class ValidationError(Exception):
    """Raised when the agent provides invalid input."""
    def __init__(self, message: str, fix_hint: str = ""):
        super().__init__(message)
        self.fix_hint = fix_hint

class BusinessRuleError(Exception):
    """Raised when a business rule prevents the operation."""
    def __init__(self, message: str, customer_message: str = ""):
        super().__init__(message)
        self.customer_message = customer_message

class PermissionDeniedError(Exception):
    """Raised when the caller lacks authorization."""
    def __init__(self, message: str, required_role: str = ""):
        super().__init__(message)
        self.required_role = required_role


@dataclass
class ToolResult:
    """Unified result wrapper for both success and error cases."""
    success: bool
    data: Optional[Any] = None
    error: Optional[ToolErrorResponse] = None

    def to_mcp_response(self) -> dict:
        if self.success:
            return {
                "isError": False,
                "content": [{"type": "text",
                             "text": json.dumps(self.data, indent=2)}]
            }
        else:
            return self.error.to_mcp_response()


print("Exception types and ToolResult class defined.")
print("Now implement the MCPToolHandler below!")

In [ ]:
# === TODO Exercise: Build an MCP Error Handler ===

class MCPToolHandler:
    """
    Wraps tool functions with structured error handling and automatic retry.

    Usage:
        handler = MCPToolHandler(max_retries=3, base_delay_ms=1000)
        result = handler.execute(my_tool_function, arg1, arg2, kwarg1=val1)
    """

    def __init__(self, max_retries: int = 3, base_delay_ms: int = 1000):
        self.max_retries = max_retries
        self.base_delay_ms = base_delay_ms
        self.execution_log: List[dict] = []  # Track all attempts for debugging

    def _classify_exception(self, exc: Exception) -> ToolErrorResponse:
        """
        Classify a caught exception into the appropriate error category.

        TODO: Implement this method. For each exception type, return the
        appropriate ToolErrorResponse using the factory functions defined
        earlier (transient_error, validation_error, business_error,
        permission_error).

        Mapping:
        - isinstance(exc, TransientError)      -> transient_error(...)
        - isinstance(exc, ValidationError)     -> validation_error(...)
            Use exc.fix_hint for the fix_hint parameter
        - isinstance(exc, BusinessRuleError)   -> business_error(...)
            Use exc.customer_message for the customer_message parameter
        - isinstance(exc, PermissionDeniedError) -> permission_error(...)
            Use exc.required_role for the required_role parameter
        - For any unknown exception             -> transient_error(...)
            with code "UNKNOWN_ERROR" and max_retries=1
        """
        # TODO: Implement error classification (~15-20 lines)
        pass

    def _should_retry(self, error: ToolErrorResponse, attempt: int) -> bool:
        """
        Decide whether to retry based on error metadata and attempt count.

        TODO: Return True if ALL of these conditions are met:
        1. error.isRetryable is True
        2. error.errorCategory is ErrorCategory.TRANSIENT
           (validation errors are retryable but need INPUT changes,
            so we do not auto-retry them here)
        3. attempt < self.max_retries
        """
        # TODO: Implement retry decision logic (~3-5 lines)
        pass

    def _get_retry_delay_ms(self, error: ToolErrorResponse,
                            attempt: int) -> int:
        """
        Calculate retry delay using exponential backoff.

        TODO: Implement exponential backoff:
        1. If error.retryAfterMs is set, use that as the base
        2. Otherwise, use self.base_delay_ms
        3. Multiply by 2^attempt for exponential backoff
        4. Add random jitter: random.randint(0, 500)
        5. Cap at 30000ms (30 seconds)

        Return the delay in milliseconds as an integer.
        """
        # TODO: Implement exponential backoff (~5-8 lines)
        pass

    def execute(self, tool_fn: Callable, *args, **kwargs) -> ToolResult:
        """
        Execute a tool function with structured error handling and retries.

        This is the main entry point. It:
        1. Calls the tool function
        2. If it succeeds, returns ToolResult(success=True, data=result)
        3. If it throws, classifies the error and decides whether to retry
        4. Logs every attempt in self.execution_log

        This method is fully implemented -- do NOT modify it.
        """
        for attempt in range(self.max_retries + 1):
            try:
                # Log the attempt
                self.execution_log.append({
                    "attempt": attempt + 1,
                    "timestamp": datetime.now().isoformat(),
                    "status": "attempting",
                })

                # Call the tool function
                result = tool_fn(*args, **kwargs)

                # Success!
                self.execution_log[-1]["status"] = "success"
                return ToolResult(success=True, data=result)

            except Exception as exc:
                # Classify the exception
                error = self._classify_exception(exc)
                if error is None:
                    # Fallback if student hasn't implemented yet
                    error = transient_error("UNIMPLEMENTED", str(exc))

                self.execution_log[-1].update({
                    "status": "error",
                    "error_category": error.errorCategory.value,
                    "error_code": error.errorCode,
                    "message": str(exc),
                })

                # Check if we should retry
                if self._should_retry(error, attempt):
                    delay_ms = self._get_retry_delay_ms(error, attempt)
                    if delay_ms is None:
                        delay_ms = self.base_delay_ms  # Fallback
                    self.execution_log[-1]["retry_delay_ms"] = delay_ms
                    print(f"  [Retry] Attempt {attempt + 1} failed "
                          f"({error.errorCode}). "
                          f"Retrying in {delay_ms}ms...")
                    time.sleep(delay_ms / 1000)
                else:
                    # Not retryable -- return the error
                    self.execution_log[-1]["final"] = True
                    return ToolResult(success=False, error=error)

        # Exhausted all retries
        return ToolResult(success=False, error=error)


print("MCPToolHandler class defined!")
print("Fill in the three TODO methods, then run the test cell below.")

In [ ]:
#@title 🎧 Listen: Solution Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_solution_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# === Reference Solution ===
# Compare your implementation with this after attempting the exercise.

class MCPToolHandlerSolution(MCPToolHandler):
    """Reference solution for the three TODO methods."""

    def _classify_exception(self, exc: Exception) -> ToolErrorResponse:
        if isinstance(exc, TransientError):
            return transient_error(
                code="TRANSIENT_FAILURE",
                message=str(exc),
                retry_after_ms=self.base_delay_ms,
                max_retries=self.max_retries,
            )
        elif isinstance(exc, ValidationError):
            return validation_error(
                code="VALIDATION_FAILURE",
                message=str(exc),
                fix_hint=exc.fix_hint,
            )
        elif isinstance(exc, BusinessRuleError):
            return business_error(
                code="BUSINESS_RULE_VIOLATION",
                message=str(exc),
                customer_message=(
                    exc.customer_message
                    or "This operation cannot be completed due to a "
                       "policy restriction."
                ),
            )
        elif isinstance(exc, PermissionDeniedError):
            return permission_error(
                code="PERMISSION_DENIED",
                message=str(exc),
                required_role=exc.required_role,
            )
        else:
            # Unknown exceptions default to transient -- safer to retry
            return transient_error(
                code="UNKNOWN_ERROR",
                message=f"Unexpected error: {type(exc).__name__}: {exc}",
                retry_after_ms=self.base_delay_ms,
                max_retries=1,
            )

    def _should_retry(self, error: ToolErrorResponse, attempt: int) -> bool:
        return (
            error.isRetryable
            and error.errorCategory == ErrorCategory.TRANSIENT
            and attempt < self.max_retries
        )

    def _get_retry_delay_ms(self, error: ToolErrorResponse,
                            attempt: int) -> int:
        base = (error.retryAfterMs
                if error.retryAfterMs
                else self.base_delay_ms)
        delay = base * (2 ** attempt)
        jitter = random.randint(0, 500)
        return min(delay + jitter, 30000)


print("Reference solution loaded. Running tests...\n")

In [ ]:
# === Test the MCPToolHandler with all 4 error types ===

print("=" * 65)
print("TEST SUITE: MCPToolHandlerSolution")
print("=" * 65)

# --- Test 1: Transient error that recovers on 3rd attempt ---
call_count_1 = 0
def flaky_refund(order_id: str, amount: float) -> dict:
    global call_count_1
    call_count_1 += 1
    if call_count_1 < 3:
        raise TransientError(
            f"Payment gateway timeout (attempt {call_count_1})")
    return {"refund_id": "REF-001", "amount": amount, "status": "processed"}

print("\n--- Test 1: Transient error (recovers on attempt 3) ---")
handler1 = MCPToolHandlerSolution(max_retries=3, base_delay_ms=200)
call_count_1 = 0
result = handler1.execute(flaky_refund, "ORD-123", 49.99)
print(f"  Success: {result.success}")
if result.success:
    print(f"  Data: {result.data}")
print(f"  Total attempts: {len(handler1.execution_log)}")
assert result.success, "Test 1 FAILED: should have succeeded on attempt 3"
print("  PASSED")

# --- Test 2: Validation error (not auto-retried) ---
def strict_email_tool(email: str) -> dict:
    if "@" not in email:
        raise ValidationError(
            "Invalid email format",
            fix_hint="Must contain @ symbol (e.g., user@example.com)")
    return {"status": "sent"}

print("\n--- Test 2: Validation error (should NOT auto-retry) ---")
handler2 = MCPToolHandlerSolution(max_retries=3, base_delay_ms=200)
result = handler2.execute(strict_email_tool, "bad-email")
print(f"  Success: {result.success}")
print(f"  Category: {result.error.errorCategory.value}")
print(f"  isRetryable: {result.error.isRetryable} (but agent must fix input)")
print(f"  Fix hint: {result.error.context.get('fix_hint', 'N/A')}")
print(f"  Attempts: {len(handler2.execution_log)} (should be 1)")
assert not result.success, "Test 2 FAILED: should have failed"
assert len(handler2.execution_log) == 1, "Test 2 FAILED: should not auto-retry"
print("  PASSED")

# --- Test 3: Business rule error ---
def strict_refund_tool(amount: float) -> dict:
    raise BusinessRuleError(
        "Refund amount exceeds order total",
        customer_message="The maximum refund for this order is $50.00. "
                         "Would you like me to process that amount?")

print("\n--- Test 3: Business rule error ---")
handler3 = MCPToolHandlerSolution(max_retries=3, base_delay_ms=200)
result = handler3.execute(strict_refund_tool, 150.00)
print(f"  Success: {result.success}")
print(f"  Category: {result.error.errorCategory.value}")
print(f"  Customer message: {result.error.customerMessage}")
assert result.error.errorCategory == ErrorCategory.BUSINESS
assert not result.error.isRetryable
print("  PASSED")

# --- Test 4: Permission error ---
def admin_only_tool() -> dict:
    raise PermissionDeniedError(
        "Requires admin access to view audit logs",
        required_role="admin")

print("\n--- Test 4: Permission error ---")
handler4 = MCPToolHandlerSolution(max_retries=3, base_delay_ms=200)
result = handler4.execute(admin_only_tool)
print(f"  Success: {result.success}")
print(f"  Category: {result.error.errorCategory.value}")
print(f"  Required role: {result.error.context.get('required_role')}")
assert result.error.errorCategory == ErrorCategory.PERMISSION
assert not result.error.isRetryable
print("  PASSED")

print("\n" + "=" * 65)
print("All 4 tests passed!")
print("=" * 65)

In [ ]:
#@title 🎧 Listen: Empty Vs Failure
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_empty_vs_failure.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---
## Section 6: Access Failures vs Valid Empty Results

This is one of the most important -- and most commonly confused -- distinctions in tool design. Consider these two scenarios:

**Scenario A:** You search for orders placed by customer "jane@example.com" and get back an empty list.

**Scenario B:** The database connection times out while searching for orders by "jane@example.com."

Both return "no results" to the agent. But they mean completely different things:

- **A** = The query succeeded. Jane has zero orders. This is a **valid empty result** (`isError: false`).
- **B** = The query could not execute. We do not know if Jane has orders. This is an **access failure** (`isError: true`).

If you treat both the same way, your agent might tell a customer "You have no orders" when really the database was down. That is a **data integrity failure**.

In [ ]:
# === Access Failures vs Valid Empty Results ===

class OrdersDatabase:
    """Mock database that can simulate both empty results and failures."""

    def __init__(self, is_healthy: bool = True):
        self.is_healthy = is_healthy
        self.data = {
            "alice@example.com": [
                {"order_id": "ORD-001", "total": 49.99,
                 "status": "delivered"},
                {"order_id": "ORD-002", "total": 129.00,
                 "status": "shipped"},
            ],
            "bob@example.com": [
                {"order_id": "ORD-003", "total": 25.00,
                 "status": "processing"},
            ],
            # jane@example.com is NOT here -- she has zero orders
        }

    def query_orders(self, customer_email: str) -> dict:
        """
        Query orders for a customer.

        CRITICAL: This method clearly distinguishes between
        'no results found' (success) and 'could not search' (error).
        """
        if not self.is_healthy:
            # ACCESS FAILURE -- we could NOT execute the query
            raise TransientError(
                "Connection to orders database timed out after 5000ms")

        # Query succeeded. Look up the customer.
        orders = self.data.get(customer_email, [])

        # Return success even when the list is empty.
        # The query WORKED. It found 0 results. That is valid.
        return {
            "isError": False,
            "resultCount": len(orders),
            "orders": orders,
            "queryExecuted": True,
            "message": f"Found {len(orders)} order(s) for {customer_email}",
        }


# === Demonstrate the critical difference ===

print("=" * 65)
print("CASE 1: Valid empty result (query succeeds, 0 orders found)")
print("=" * 65)

db_healthy = OrdersDatabase(is_healthy=True)
result = db_healthy.query_orders("jane@example.com")
print(f"\nResponse: {json.dumps(result, indent=2)}")
print("\nAgent interpretation:")
print("  isError: False  --> The query executed successfully")
print("  resultCount: 0  --> Jane genuinely has no orders")
print("  Correct action:  Tell Jane 'You don't have any orders yet.'")

print(f"\n{'=' * 65}")
print("CASE 2: Access failure (query could NOT execute)")
print("=" * 65)

db_down = OrdersDatabase(is_healthy=False)
handler = MCPToolHandlerSolution(max_retries=1, base_delay_ms=200)
result = handler.execute(db_down.query_orders, "jane@example.com")
print(f"\nResponse: {json.dumps(result.to_mcp_response(), indent=2)}")
print("\nAgent interpretation:")
print("  isError: True   --> The query FAILED")
print("  Category: transient --> Database timeout")
print("  Correct action:  DO NOT tell Jane she has no orders!")
print("                   Retry or escalate.")

print(f"\n{'=' * 65}")
print("CASE 3: Successful query with results (for comparison)")
print("=" * 65)

result = db_healthy.query_orders("alice@example.com")
print(f"\nResponse: {json.dumps(result, indent=2)}")
print("\nAgent interpretation:")
print("  isError: False, resultCount: 2 --> Show Alice her 2 orders")

In [ ]:
# === Automated Tests: Prove the Distinction is Enforced ===

def test_empty_vs_failure():
    """
    These tests codify the contract:
    - Empty results MUST have isError=False
    - Access failures MUST have isError=True
    - They must NEVER be confused
    """
    passed = 0
    failed = 0

    def check(name: str, condition: bool, detail: str = ""):
        nonlocal passed, failed
        if condition:
            passed += 1
            print(f"  PASS: {name}")
        else:
            failed += 1
            print(f"  FAIL: {name} -- {detail}")

    print("Running access failure vs empty result tests...\n")

    # --- Test group 1: Empty result is NOT an error ---
    db = OrdersDatabase(is_healthy=True)
    result = db.query_orders("nobody@example.com")

    check("Empty result has isError=False",
          result["isError"] == False,
          f"Expected False, got {result['isError']}")

    check("Empty result has resultCount=0",
          result["resultCount"] == 0,
          f"Expected 0, got {result['resultCount']}")

    check("Empty result has queryExecuted=True",
          result["queryExecuted"] == True,
          "Query DID execute, should be True")

    # --- Test group 2: Non-empty result is also not an error ---
    result = db.query_orders("alice@example.com")

    check("Non-empty result has isError=False",
          result["isError"] == False)

    check("Non-empty result has correct count",
          result["resultCount"] == 2,
          f"Expected 2, got {result['resultCount']}")

    # --- Test group 3: Access failure IS an error ---
    db_down = OrdersDatabase(is_healthy=False)
    handler = MCPToolHandlerSolution(max_retries=0, base_delay_ms=100)
    result = handler.execute(db_down.query_orders, "jane@example.com")

    check("Access failure returns success=False",
          result.success == False)

    check("Access failure has isError=True in MCP response",
          result.to_mcp_response()["isError"] == True)

    check("Access failure is categorized as transient",
          result.error.errorCategory == ErrorCategory.TRANSIENT)

    # --- Test group 4: THE CRITICAL CHECK ---
    db_up = OrdersDatabase(is_healthy=True)
    empty_result = db_up.query_orders("nobody@example.com")

    db_down = OrdersDatabase(is_healthy=False)
    handler = MCPToolHandlerSolution(max_retries=0, base_delay_ms=100)
    failure_result = handler.execute(
        db_down.query_orders, "nobody@example.com")

    check(
        "CRITICAL: Empty result (isError=False) != "
        "Access failure (isError=True)",
        empty_result["isError"] != failure_result.to_mcp_response()["isError"],
        "These MUST be different!")

    print(f"\n{'=' * 50}")
    print(f"Results: {passed} passed, {failed} failed out of {passed+failed}")
    if failed == 0:
        print("All tests passed!")
    print(f"{'=' * 50}")


test_empty_vs_failure()

In [ ]:
#@title 🎧 Listen: Multiagent Project
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_multiagent_project.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

**Why this matters on the exam:**

| Confusion | Consequence |
|---|---|
| Treating access failure as empty result | Agent tells customer "no orders found" when database was down. **Data integrity violation.** |
| Treating empty result as access failure | Agent says "I'm having trouble looking that up" when the customer genuinely has no orders. **Unnecessary escalation and confusion.** |

**The rule:** If the tool could not confirm what the ground truth is, it must signal `isError: true`. Only return `isError: false` when the tool is confident in its result -- even if that result is empty.

---
## Section 7: Mini-Project -- Local Error Recovery in Subagents

Now let's put everything together. In a real production system, you often have **multiple agents** coordinating work. A coordinator agent delegates tasks to specialized subagents, each of which calls its own tools.

The question is: **who handles which errors?**

| Error Type | Who Handles | Why |
|---|---|---|
| **Transient** | Subagent (local) | Infrastructure retry is self-contained. The coordinator does not need to know. |
| **Validation** | Subagent (local) | The subagent made the bad call; it should fix its own input. |
| **Business** | Coordinator (propagated) | The coordinator needs to explain to the user or adjust overall strategy. |
| **Permission** | Coordinator (propagated) | The coordinator needs to escalate or reassign to a different agent. |

In [ ]:
# === Mini-Project: Multi-Agent System with Error Propagation ===

# ---- Mock Services with Realistic Error Profiles ----

class InventoryService:
    """Simulates an inventory checking service."""

    def __init__(self, failure_mode: str = "none"):
        """
        failure_mode: "none", "transient", "out_of_stock", "restricted"
        """
        self.failure_mode = failure_mode
        self.call_count = 0

    def check_stock(self, product_id: str, quantity: int) -> dict:
        self.call_count += 1

        if self.failure_mode == "transient" and self.call_count <= 2:
            raise TransientError(
                f"Inventory service timeout (attempt {self.call_count})")

        if self.failure_mode == "out_of_stock":
            raise BusinessRuleError(
                f"Product {product_id} has 0 units in stock",
                customer_message=(
                    f"Sorry, {product_id} is currently out of stock. "
                    f"Would you like to be notified when it's back?"))

        if self.failure_mode == "restricted":
            raise PermissionDeniedError(
                f"Product {product_id} requires age verification",
                required_role="age_verified_agent")

        return {
            "product_id": product_id,
            "available": quantity + 10,
            "warehouse": "US-WEST-2",
        }


class PaymentService:
    """Simulates a payment processing service."""

    def __init__(self, failure_mode: str = "none"):
        self.failure_mode = failure_mode
        self.call_count = 0

    def charge(self, customer_id: str, amount: float) -> dict:
        self.call_count += 1

        if self.failure_mode == "transient" and self.call_count <= 1:
            raise TransientError(
                "Payment gateway rate limited. Retry in 1s.")

        if self.failure_mode == "insufficient_funds":
            raise BusinessRuleError(
                "Card ending 4242 declined: insufficient funds",
                customer_message=(
                    "Your card was declined. Would you like to try "
                    "a different payment method?"))

        return {
            "charge_id": f"CHG-{random.randint(1000, 9999)}",
            "amount": amount,
            "status": "charged",
        }


print("Mock services defined: InventoryService, PaymentService")

In [ ]:
# ---- Subagent: Handles local recovery ----

@dataclass
class SubagentResult:
    """Result from a subagent, including propagation metadata."""
    success: bool
    data: Optional[Any] = None
    error: Optional[ToolErrorResponse] = None
    agent_name: str = ""
    attempts_made: int = 0
    locally_recovered: bool = False

    def needs_coordinator(self) -> bool:
        """Does this error need to be handled by the coordinator?"""
        if self.success:
            return False
        return self.error.errorCategory in (
            ErrorCategory.BUSINESS,
            ErrorCategory.PERMISSION,
        )


class Subagent:
    """
    A specialized agent that handles one domain (inventory, payment).

    Key behavior:
    - Transient errors: retry silently (local recovery)
    - Validation errors: fix input silently (local recovery)
    - Business/Permission errors: propagate to coordinator with full context
    """

    def __init__(self, name: str, max_retries: int = 3):
        self.name = name
        self.max_retries = max_retries
        self.log: List[str] = []

    def _log(self, msg: str):
        entry = f"[{self.name}] {msg}"
        self.log.append(entry)
        print(f"    {entry}")

    def execute_tool(self, tool_fn: Callable, *args, **kwargs) -> SubagentResult:
        """Execute a tool with local error recovery."""
        self._log(f"Calling {tool_fn.__name__}({args})")

        handler = MCPToolHandlerSolution(
            max_retries=self.max_retries,
            base_delay_ms=300,
        )
        result = handler.execute(tool_fn, *args, **kwargs)
        attempts = len(handler.execution_log)

        if result.success:
            recovered = attempts > 1
            if recovered:
                self._log(f"Succeeded after {attempts} attempts "
                          f"(recovered locally)")
            else:
                self._log(f"Succeeded on first attempt")
            return SubagentResult(
                success=True,
                data=result.data,
                agent_name=self.name,
                attempts_made=attempts,
                locally_recovered=recovered,
            )
        else:
            error = result.error
            cat = error.errorCategory.value
            self._log(f"FAILED: [{cat}] {error.message}")

            if error.errorCategory in (ErrorCategory.BUSINESS,
                                       ErrorCategory.PERMISSION):
                self._log(f"Propagating {cat} error to coordinator")
            else:
                self._log(f"Exhausted {attempts} retry attempts")

            return SubagentResult(
                success=False,
                error=error,
                agent_name=self.name,
                attempts_made=attempts,
            )


print("Subagent class defined.")

In [ ]:
# ---- Coordinator Agent: Orchestrates subagents ----

class CoordinatorAgent:
    """
    Top-level agent that orchestrates subagents for order processing.
    Handles propagated errors with user-facing responses.
    """

    def __init__(self):
        self.inventory_agent = Subagent("InventoryAgent", max_retries=3)
        self.payment_agent = Subagent("PaymentAgent", max_retries=2)

    def _say(self, msg: str):
        """What the coordinator says to the user."""
        print(f"\n  --> TO CUSTOMER: \"{msg}\"")

    def _think(self, msg: str):
        """Internal reasoning (visible for debugging)."""
        print(f"  [Coordinator thinking] {msg}")

    def process_order(
        self,
        customer_id: str,
        product_id: str,
        quantity: int,
        amount: float,
        inventory_svc: InventoryService,
        payment_svc: PaymentService,
    ) -> dict:
        """
        Full order processing workflow:
        1. Check inventory (InventoryAgent)
        2. Process payment (PaymentAgent)
        3. Confirm order

        Subagents handle transient errors locally.
        Business/permission errors propagate here for user-facing decisions.
        """
        print(f"\n{'=' * 65}")
        print(f"ORDER: {customer_id} wants {quantity}x {product_id} "
              f"for ${amount:.2f}")
        print(f"{'=' * 65}")

        # === Step 1: Check inventory ===
        self._think("Step 1: Checking inventory...")
        inv_result = self.inventory_agent.execute_tool(
            inventory_svc.check_stock, product_id, quantity)

        if not inv_result.success:
            cat = inv_result.error.errorCategory

            if cat == ErrorCategory.BUSINESS:
                self._say(inv_result.error.customerMessage)
                return {
                    "status": "failed",
                    "stage": "inventory",
                    "reason": inv_result.error.errorCode,
                    "customer_message": inv_result.error.customerMessage,
                }
            elif cat == ErrorCategory.PERMISSION:
                self._say(inv_result.error.customerMessage)
                role = inv_result.error.context.get("required_role", "unknown")
                self._think(f"Need to escalate to role: {role}")
                return {
                    "status": "failed",
                    "stage": "inventory",
                    "reason": "permission",
                    "escalate_to": role,
                }
            else:
                self._say("I'm having trouble checking availability. "
                          "Let me try again in a moment.")
                return {
                    "status": "failed",
                    "stage": "inventory",
                    "reason": "service_unavailable",
                }

        if inv_result.locally_recovered:
            self._think("Inventory had a hiccup but recovered. "
                        "Customer does not need to know.")

        # === Step 2: Process payment ===
        self._think("Step 2: Processing payment...")
        pay_result = self.payment_agent.execute_tool(
            payment_svc.charge, customer_id, amount)

        if not pay_result.success:
            cat = pay_result.error.errorCategory

            if cat == ErrorCategory.BUSINESS:
                self._say(pay_result.error.customerMessage)
                return {
                    "status": "failed",
                    "stage": "payment",
                    "reason": pay_result.error.errorCode,
                    "customer_message": pay_result.error.customerMessage,
                }
            else:
                self._say("Payment processing is temporarily unavailable. "
                          "Your order is saved and we'll retry shortly.")
                return {
                    "status": "failed",
                    "stage": "payment",
                    "reason": "service_unavailable",
                }

        # === Step 3: Confirm order ===
        self._think("Both steps succeeded! Confirming order.")
        order_id = f"ORD-{random.randint(10000, 99999)}"
        self._say(f"Your order {order_id} is confirmed! "
                  f"{quantity}x {product_id} for ${amount:.2f}.")

        return {
            "status": "success",
            "order_id": order_id,
            "inventory": inv_result.data,
            "payment": pay_result.data,
            "recovery_stats": {
                "inventory_attempts": inv_result.attempts_made,
                "payment_attempts": pay_result.attempts_made,
                "inventory_recovered_locally": inv_result.locally_recovered,
                "payment_recovered_locally": pay_result.locally_recovered,
            },
        }


print("CoordinatorAgent defined. Running 4 scenarios below...")

In [ ]:
#@title 🎧 Listen: Scenarios Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_scenarios_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# === Scenario A: Happy path with transient recovery ===
# Inventory flakes twice, then succeeds. Payment flakes once, then succeeds.
# The user should NEVER know about the retries.

print("\n" + "#" * 65)
print("# SCENARIO A: Transient errors with local recovery")
print("# (user never knows there were problems)")
print("#" * 65)

coordinator = CoordinatorAgent()
result = coordinator.process_order(
    customer_id="CUST-001",
    product_id="WIDGET-X",
    quantity=2,
    amount=39.98,
    inventory_svc=InventoryService(failure_mode="transient"),
    payment_svc=PaymentService(failure_mode="transient"),
)

print(f"\nFinal result: {json.dumps(result, indent=2)}")
print("\nKey insight: The user saw a smooth experience.")
print("Both subagents recovered from transient errors locally.")

In [ ]:
# === Scenario B: Business error propagated to coordinator ===
# Product is out of stock. Subagent cannot fix this -- propagates immediately.

print("\n" + "#" * 65)
print("# SCENARIO B: Business error -- propagated to coordinator")
print("#" * 65)

coordinator = CoordinatorAgent()
result = coordinator.process_order(
    customer_id="CUST-002",
    product_id="RARE-ITEM-99",
    quantity=1,
    amount=299.99,
    inventory_svc=InventoryService(failure_mode="out_of_stock"),
    payment_svc=PaymentService(failure_mode="none"),
)

print(f"\nFinal result: {json.dumps(result, indent=2)}")
print("\nKey insight: The subagent did NOT retry (business errors are")
print("not retryable). It propagated immediately with a customer message.")

In [ ]:
# === Scenario C: Permission error requiring escalation ===
# Age-restricted product. Current agent lacks the required role.

print("\n" + "#" * 65)
print("# SCENARIO C: Permission error -- requires escalation")
print("#" * 65)

coordinator = CoordinatorAgent()
result = coordinator.process_order(
    customer_id="CUST-003",
    product_id="AGE-RESTRICTED-ITEM",
    quantity=1,
    amount=19.99,
    inventory_svc=InventoryService(failure_mode="restricted"),
    payment_svc=PaymentService(failure_mode="none"),
)

print(f"\nFinal result: {json.dumps(result, indent=2)}")
print("\nKey insight: Permission errors propagate with the required_role")
print("field. The coordinator knows exactly who to escalate to.")

In [ ]:
# === Scenario D: Payment declined after successful inventory check ===
# Inventory succeeds, but payment hits a business rule violation.

print("\n" + "#" * 65)
print("# SCENARIO D: Payment declined (business error at step 2)")
print("#" * 65)

coordinator = CoordinatorAgent()
result = coordinator.process_order(
    customer_id="CUST-004",
    product_id="WIDGET-X",
    quantity=1,
    amount=49.99,
    inventory_svc=InventoryService(failure_mode="none"),
    payment_svc=PaymentService(failure_mode="insufficient_funds"),
)

print(f"\nFinal result: {json.dumps(result, indent=2)}")
print("\nKey insight: The system correctly completed step 1 before failing")
print("at step 2. The payment subagent propagated the business error with")
print("a customer message that offers an alternative payment method.")

In [ ]:
# === Summary: Error Propagation Matrix ===

print("=" * 70)
print("ERROR PROPAGATION MATRIX -- The Pattern to Remember")
print("=" * 70)
print()
headers = f"{'Category':<15} {'Subagent Action':<25} {'Propagated?':<14} {'Coordinator Action'}"
print(headers)
print("-" * 80)
print(f"{'TRANSIENT':<15} {'Retry with backoff':<25} {'No':<14} {'N/A (handled locally)'}")
print(f"{'VALIDATION':<15} {'Fix input, retry':<25} {'No':<14} {'N/A (handled locally)'}")
print(f"{'BUSINESS':<15} {'Do NOT retry':<25} {'Yes':<14} {'Explain + alternatives'}")
print(f"{'PERMISSION':<15} {'Do NOT retry':<25} {'Yes':<14} {'Escalate to right role'}")
print()
print("The rule:")
print("  If a subagent CAN fix it alone   --> handle locally")
print("  If a subagent CANNOT fix it       --> propagate with full context")
print()
print("This is the architectural pattern tested on the Architect exam.")
print("You are expected to know WHICH errors propagate and WHY.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---
## Section 8: Key Takeaways + What's Next

### What You Learned

1. **The `isError` flag** is your first-line signal in MCP tool responses. Without it, agents are flying blind.

2. **Four error categories** cover virtually all tool failures:
   - **Transient** -- infrastructure flakes, retry with backoff
   - **Validation** -- bad input, fix and retry
   - **Business** -- policy constraint, explain to user
   - **Permission** -- authorization failure, escalate

3. **`isRetryable` alone is not enough.** You also need to know *how* to retry:
   - Transient: identical request, just wait and try again
   - Validation: modified request, fix the input first
   - Business/Permission: do not retry at all

4. **Access failures vs empty results** are categorically different:
   - Empty result: `isError: false` -- the query succeeded, found 0 items
   - Access failure: `isError: true` -- the query could not execute, result unknown

5. **Multi-agent error propagation** follows a clear rule:
   - Subagents handle transient/validation errors **locally** (silent recovery)
   - Business/permission errors **propagate** to the coordinator with full context

### Architect Exam Pointers

- Expect to classify error scenarios into the four categories
- Know the difference between "retryable" and "auto-retryable" (validation is retryable but NOT auto-retryable)
- The empty-result-vs-access-failure distinction appears frequently in exam questions
- Multi-agent propagation patterns are tested in system design sections

### What's Next

In **Notebook 3: Tool Distribution & `tool_choice`**, you will learn how to:
- Scope tools to specific agents (not every agent needs every tool)
- Use `auto`, `any`, and forced `tool_choice` to control tool selection
- Prevent cross-specialization misuse (e.g., a billing agent calling an inventory tool)
- Build a document processing pipeline with enforced tool ordering

That notebook builds directly on the error handling patterns from this notebook -- tool scoping is another layer of defense against misuse and confusion.